In [1]:
from quant_risk.setup import base, asset_pricing, macro

np, pd, plt = base()
ois_curve, nss_curve, valuation_date, calendar, ql = asset_pricing()
fed_client, store, external_store = macro()

base loaded
OISCurve | currency=EUR | valuation_date=2026-03-24
NSSCurve | currency=EUR | valuation_date=2026-05-04
Valuation date : March 24th, 2026
Calendar       : TARGET
asset pricing loaded
macro loaded


## 0 FX Rate Conventions

An FX rate is expressed as units of the **quote currency**
needed to buy one unit of the **base currency**:

$$\text{BASE/QUOTE} = \text{price of 1 BASE in QUOTE units}$$

**Market conventions:**

| Pair | Base | Quote | Convention |
|------|------|-------|------------|
| EUR/USD | EUR | USD | EUR always base vs USD |
| EUR/CHF | EUR | CHF | EUR base vs Swiss franc |
| GBP/USD | GBP | USD | GBP always base vs USD |
| USD/JPY | USD | JPY | USD base vs Asian currencies |
| USD/CHF | USD | CHF | USD base vs Swiss franc |
| USD/BRL | USD | BRL | USD base vs EM currencies |
| EUR/GBP | EUR | GBP | EUR base vs other European currencies |

EUR is the strongest base currency -- it is quoted as base against
almost all other currencies. USD is base against most Asian and
emerging market currencies but quote against EUR and GBP. CHF appears
on both sides depending on the pair -- EUR/CHF and USD/CHF both have
CHF as quote, reflecting the Swiss franc's role as a safe-haven
funding currency rather than a base.


For this notebook we price **EUR/USD** forwards, the most liquid
FX pair globally, with approximately USD 1.5 trillion traded daily.

# Asset Pricing: FX Forwards

An FX forward is an agreement to exchange two currencies at a
predetermined rate on a future date. No cash changes hands at
inception -- the exchange occurs at maturity at the agreed forward rate.

**Covered Interest Rate Parity (CIP):**

The forward rate is not a forecast -- it is an arbitrage condition.
If the forward rate deviated from CIP, a riskless profit would be
available by borrowing in one currency, converting spot, investing in
the other currency, and locking in the forward. In equilibrium:

$$F(0,T) = S_0 \cdot \frac{P_{EUR}(0,T)}{P_{USD}(0,T)}$$

Where:
- $F(0,T)$ -- forward EUR/USD rate for delivery at T
- $S_0$ -- spot EUR/USD rate today
- $P_{EUR}(0,T)$ -- EUR OIS discount factor to maturity T
- $P_{USD}(0,T)$ -- USD OIS (SOFR) discount factor to maturity T

In continuous compounding:

$$F(0,T) = S_0 \cdot e^{(r_{USD} - r_{EUR}) \cdot T}$$

The forward rate is higher than spot when USD rates exceed EUR rates.

**CIP deviation -- the basis:**

Post-2008 CIP does not hold exactly in practice. The deviation is
called the **cross-currency basis** -- the spread paid above or below
CIP to obtain funding in a foreign currency via the FX swap market.
A negative EUR/USD basis means it costs more to borrow USD via EUR
FX swaps than the CIP rate implies -- reflecting USD funding scarcity.

**Regulation context:**
- FRTB SA -- delta FX risk sensitivity per currency pair
- EMIR -- FX forwards with maturity > 3 days are OTC derivatives
  subject to reporting and margin requirements
- IFRS 9 -- FX forwards are Level 2 fair value instruments

IFRS 13 defines three fair value levels:

  <span style="color:#2ca02c">- **Level 1** -- quoted prices in active markets (listed equities, exchange-traded futures)

  <span style="color:#2ca02c">- **Level 2** -- observable inputs other than Level 1 (OTC derivatives, FX forwards, interest rate swaps -- priced from market curves)

  <span style="color:#2ca02c">- **Level 3** -- unobservable inputs, significant model judgment (illiquid structured products, private equity, complex exotics)

## 1 FX and <span style="color:#0c9000">CIP (Covered Interest Rate Parity)

Consider a EUR-based investor with EUR 1M to invest for 1 year.
Two strategies must produce the same result in equilibrium:

**Strategy A -- invest in EUR:**
- Invest EUR 1M at the EUR risk-free rate $r_{EUR}$
- Receive EUR $1M \cdot e^{r_{EUR} \cdot T}$ at maturity

**Strategy B -- invest in USD:**
- Convert EUR 1M to USD at spot rate $S_0$: receive USD $1M \cdot S_0$
- Invest USD at the USD risk-free rate $r_{USD}$
- Receive USD $1M \cdot S_0 \cdot e^{r_{USD} \cdot T}$ at maturity
- Convert back to EUR at the forward rate $F(0,T)$
- Receive EUR $1M \cdot S_0 \cdot e^{r_{USD} \cdot T} / F(0,T)$

**No-arbitrage condition:**

Both strategies must produce the same EUR amount:

$$e^{r_{EUR} \cdot T} = S_0 \cdot e^{r_{USD} \cdot T} / F(0,T)$$

Solving for the forward rate:

$$F(0,T) = S_0 \cdot e^{(r_{USD} - r_{EUR}) \cdot T}$$

**What this means in practice:**

If USD rates are higher than EUR rates ($r_{USD} > r_{EUR}$), the
forward rate $F(0,T)$ is higher than the spot rate $S_0$ -- the USD
trades at a forward discount (you need more USD per EUR in the future).
If EUR rates are higher, the forward is below spot.

The forward rate is therefore not a market forecast of where EUR/USD
will be in one year -- it is a mathematical consequence of today's
interest rates. Any deviation would create a riskless arbitrage.

**Example with current rates:**

If EUR OIS 1Y = 2.00% and USD SOFR 1Y = 4.50% and spot = 1.0800:

$$F(0,1Y) = 1.0800 \cdot e^{(0.045 - 0.020) \cdot 1} = 1.0800 \cdot e^{0.025} = 1.1072$$

The EUR trades at a forward premium -- you get more USD per EUR in
one year because USD rates are higher and the market compensates EUR
holders for the lower EUR investment return.

## 2 Market Data -- Aligning to OIS Valuation Date

The EUR OIS curve was bootstrapped from ECB MMSR data observed on
**March 24 2026** -- actual executed transactions in the euro overnight
swap market. To ensure consistency we align all market data to this
same date:

- **EUR OIS curve** -- loaded from `data/processed/`, valuation date March 24 2026
- **USD SOFR overnight** -- fetched from FRED, nearest available fixing to March 24 2026
- **USD Treasury curve** -- fetched from FRED, nearest available observation to March 24 2026
- **EUR/USD spot rate** -- fetched from FRED, nearest available fixing to March 24 2026

In production all curves would be built from same-day market data.
Here we use the ECB MMSR date as the anchor and fetch the closest
available USD data from FRED.

In [4]:
# fetch USD data for the same date as OIS curve -- March 24 2026
target_date = ois_curve.valuation_date  # "2026-03-24"

sofr_series    = fed_client.get_overnight_rate(last_n=252)
treasury_curve = fed_client.get_full_curve(last_n=252)
eurusd_series  = fed_client.get_fx_spot("EURUSD", last_n=252)

# get values for target date or nearest prior date
def get_at_date(series, target):
    if target in series.index:
        return series[target]
    prior = series[series.index <= target]
    if prior.empty:
        raise ValueError(f"No data available on or before {target}")
    print(f"  {series.name} -- using {prior.index[-1]} (target {target} not available)")
    return prior.iloc[-1]

sofr_overnight = get_at_date(sofr_series, target_date)
spot_eurusd    = get_at_date(eurusd_series, target_date)

# treasury curve at target date
treasury_row = None
dates_available = treasury_curve.index
prior_dates = [d for d in dates_available if d <= target_date]
if target_date in dates_available:
    treasury_row = treasury_curve.loc[target_date]
else:
    nearest = max(prior_dates)
    treasury_row = treasury_curve.loc[nearest]
    print(f"  Treasury curve -- using {nearest} (target {target_date} not available)")

print(f"\nAll data aligned to: {target_date}")
print(f"SOFR overnight : {sofr_overnight:.4f}%")
print(f"Spot EUR/USD   : {spot_eurusd:.4f}")
print(f"\nUS Treasury curve:")
print(treasury_row)


All data aligned to: 2026-03-24
SOFR overnight : 3.6300%
Spot EUR/USD   : 1.1578

US Treasury curve:
3M    3.740000
6M    3.780000
1Y    3.810000
2Y    3.900000
5Y    4.030000
10Y   4.390000
20Y   4.950000
30Y   4.940000
Name: 2026-03-24, dtype: float64


In [5]:
# map Treasury maturities to years
treasury_tenors = {
    "3M": 0.25, "6M": 0.5, "1Y": 1.0, "2Y": 2.0,
    "5Y": 5.0, "10Y": 10.0, "20Y": 20.0, "30Y": 30.0
}

us_calendar  = ql.UnitedStates(ql.UnitedStates.GovernmentBond)
day_count_fx = ql.Actual360()

# build USD curve from aligned FRED data
usd_dates = [valuation_date] + [
    valuation_date + ql.Period(max(1, int(t * 365)), ql.Days)
    for t in treasury_tenors.values()
]
usd_rates = [sofr_overnight / 100] + [
    treasury_row[m] / 100
    for m in treasury_tenors.keys()
]

usd_zc = ql.ZeroCurve(
    usd_dates, usd_rates, day_count_fx,
    us_calendar, ql.Linear(), ql.Continuous
)
usd_zc.enableExtrapolation()
usd_handle = ql.YieldTermStructureHandle(usd_zc)

# save pillar rates for MTM reuse
usd_pillar_dates = usd_dates.copy()
usd_pillar_rates = usd_rates.copy()

print(f"\n{'Tenor':<8} {'EUR OIS':>10} {'USD Treasury':>14} {'Differential':>14}")
print("-" * 50)
for m, t in treasury_tenors.items():
    eur_r = ois_curve.zero_rate(t)
    usd_r = treasury_row[m]
    diff  = usd_r - eur_r
    print(f"{m:<8} {eur_r:>10.4f}% {usd_r:>13.4f}% {diff:>13.4f}%")


Tenor       EUR OIS   USD Treasury   Differential
--------------------------------------------------
3M           1.9786%        3.7400%        1.7614%
6M           2.0521%        3.7800%        1.7279%
1Y           2.0026%        3.8100%        1.8074%
2Y           2.1312%        3.9000%        1.7688%
5Y           2.3096%        4.0300%        1.7204%
10Y          2.6167%        4.3900%        1.7733%
20Y          3.1814%        4.9500%        1.7686%
30Y          3.3977%        4.9400%        1.5423%


## 3 Price Discovery -- Forward Rate at CIP

In [6]:
# build EUR OIS ZeroCurve for discount factor queries
eur_tenors   = [0.0, 1/12, 2/12, 3/12, 6/12, 9/12, 1, 2, 3, 5, 10, 15]
eur_dates_fx = [
    valuation_date if t == 0
    else valuation_date + ql.Period(max(1, int(t * 365)), ql.Days)
    for t in eur_tenors
]
eur_rates_fx = [
    ois_curve.zero_rate(1/365) / 100 if t == 0
    else ois_curve.zero_rate(t) / 100
    for t in eur_tenors
]

eur_zc = ql.ZeroCurve(
    eur_dates_fx, eur_rates_fx, day_count_fx,
    ql.TARGET(), ql.Linear(), ql.Continuous
)
eur_zc.enableExtrapolation()
eur_handle = ql.YieldTermStructureHandle(eur_zc)

# save pillar rates for MTM reuse
eur_pillar_dates = eur_dates_fx.copy()
eur_pillar_rates = eur_rates_fx.copy()

print("EUR OIS ZeroCurve built")

EUR OIS ZeroCurve built


In [7]:
# -----------------------------------------------------------------------
# section 3 -- forward rate at CIP
# -----------------------------------------------------------------------

# forward maturities to price
forward_tenors = {
    "1M"  : 1/12,
    "3M"  : 3/12,
    "6M"  : 6/12,
    "1Y"  : 1.0,
    "2Y"  : 2.0,
    "5Y"  : 5.0,
}

print(f"Spot EUR/USD     : {spot_eurusd:.4f}")
print(f"Valuation date   : {valuation_date}")
print(f"\nCIP forward rates -- F = S * P_EUR(0,T) / P_USD(0,T)")
print(f"\n{'Tenor':<8} {'EUR df':>10} {'USD df':>10} {'Forward':>10} {'Fwd pts':>10}")
print("-" * 52)

forward_rates = {}
for label, T in forward_tenors.items():
    df_eur = eur_zc.discount(
        valuation_date + ql.Period(max(1, int(T * 365)), ql.Days)
    )
    df_usd = usd_zc.discount(
        valuation_date + ql.Period(max(1, int(T * 365)), ql.Days)
    )
    fwd = spot_eurusd * df_eur / df_usd  
    fwd_points = (fwd - spot_eurusd) * 10_000
    forward_rates[label] = fwd
    print(f"{label:<8} {df_eur:>10.6f} {df_usd:>10.6f} "
          f"{fwd:>10.4f} {fwd_points:>10.1f}")

print(f"\nCIP forward rates -- F = S * P_EUR(0,T) / P_USD(0,T)")

Spot EUR/USD     : 1.1578
Valuation date   : March 24th, 2026

CIP forward rates -- F = S * P_EUR(0,T) / P_USD(0,T)

Tenor        EUR df     USD df    Forward    Fwd pts
----------------------------------------------------
1M         0.998387   0.996949     1.1595       16.7
3M         0.995011   0.990591     1.1630       51.7
6M         0.989679   0.981071     1.1680      101.6
1Y         0.979900   0.962107     1.1792      214.1
2Y         0.957705   0.923963     1.2001      422.8
5Y         0.889509   0.815219     1.2633     1055.1

CIP forward rates -- F = S * P_EUR(0,T) / P_USD(0,T)


## 4 Pricing a Seasoned FX Forward

Signs in FX forwards flip depending on whether you are receiving or
paying the base currency. To fix the intuition we adopt the
**exporter hedger perspective** throughout, the natural position
of anyone receiving foreign cash flows and wanting to lock in the
local currency value, to show how the MTM works. bellwo we show close before aturity but the logic holds the same.

| Time | Trade | Local | Foreign |
|------|-------|-------|---------|
| 0 | Sign forward -- deliver foreign, receive local at $F_0$ | $0$ | $0$ |
| T | Original forward settles | $+N_f \times F_0$ | $-N_f$ |
| t | Close -- opposite trade at current market rate $F_t$ | $-N_f \times F_t$ | $+N_f$ |
| T | **Net cash flow** | $N_f \times (F_0 - F_t)$ | $0$ |

Present value of the net local currency cash flow:

$${NPV_{local} = N_f \cdot (F_0 - F_t) \cdot P_{local}(0,T)}$$

If $F_0 > F_t$: local currency has weakened since inception -- you
gain, you locked in more local per foreign than the market now offers.

If $F_0 < F_t$: local currency has strengthened -- you lose, the
market now offers more local per foreign than your contractual rate.

## 5 Delta FX and Delta IR Sensitivity

Under FRTB SA (CRR3 Article 325) FX forwards have two delta risk factors:

**Delta FX** -- sensitivity of NPV to a 1% move in the spot rate.
The spot rate appears directly in the forward rate via CIP:
$F_t = S_t \cdot P_{EUR}/P_{USD}$, so when spot rises (EUR strengthens)
the forward rate rises, $F_0 - F_t$ falls, and the exporter loses.

$$\Delta_{FX} = \frac{\partial NPV}{\partial S} = -N_f \cdot \frac{F_0}{S_0} \cdot P_{local}(0,T)$$

Negative -- the exporter loses when EUR strengthens (spot rises).

**Delta IR** -- sensitivity of NPV to a 1bp move in the interest rate
differential $(r_{USD} - r_{EUR})$. When the differential widens,
$F_t$ rises (larger forward premium), $F_0 - F_t$ falls -- exporter loses.

$$\Delta_{IR} = \frac{\partial NPV}{\partial (r_{USD} - r_{EUR})} \approx -N_f \cdot F_0 \cdot T \cdot P_{local}(0,T) \cdot 0.0001$$

Negative -- the exporter loses when USD rates rise relative to EUR rates,
because the market now offers a better rate than the contractual $F_0$.

Under FRTB SA delta FX feeds into the FX risk bucket and delta IR
feeds into the GIRR bucket -- both contribute to capital requirements.

In [8]:
# -----------------------------------------------------------------------
# section 5 -- delta FX and delta IR sensitivity
# -----------------------------------------------------------------------

# define the forward contract
notional_usd   = 1_000_000   # USD 1M -- foreign notional
maturity_label = "1Y"
T              = forward_tenors[maturity_label]
F_0            = forward_rates[maturity_label]  # contractual rate from section 3

print(f"Forward contract: deliver USD {notional_usd:,.0f}, receive EUR")
print(f"Maturity        : {maturity_label}")
print(f"Contractual rate: {F_0:.4f} EUR/USD")

# current discount factors
dt      = valuation_date + ql.Period(max(1, int(T * 365)), ql.Days)
df_eur  = eur_zc.discount(dt)
df_usd  = usd_zc.discount(dt)
F_t     = spot_eurusd * df_eur / df_usd  # current market forward

# baseline NPV -- exporter perspective (deliver USD, receive EUR)
npv_base = notional_usd * (F_0 - F_t) * df_eur
print(f"\nCurrent market forward : {F_t:.4f}")
print(f"NPV (EUR)              : {npv_base:,.2f}")

# -----------------------------------------------------------------------
# delta FX -- bump spot by 1%
# -----------------------------------------------------------------------
spot_bump   = spot_eurusd * 0.01  # 1% of spot
spot_up     = spot_eurusd + spot_bump
F_t_up      = spot_up * df_eur / df_usd
npv_spot_up = notional_usd * (F_0 - F_t_up) * df_eur
delta_fx    = npv_spot_up - npv_base

# analytical delta FX
delta_fx_analytical = -notional_usd * (F_0 / spot_eurusd) * df_eur * 0.01

print(f"\nDelta FX (numerical, 1% spot move) : EUR {delta_fx:,.2f}")
print(f"Delta FX (analytical)              : EUR {delta_fx_analytical:,.2f}")

# -----------------------------------------------------------------------
# delta IR -- bump USD-EUR differential by 1bp
# -----------------------------------------------------------------------
rate_bump = 0.0001  # 1bp

# bump USD rates up by 1bp -- rebuild USD curve
usd_rates_up = [r + rate_bump for r in usd_pillar_rates]
usd_zc_up    = ql.ZeroCurve(
    usd_pillar_dates, usd_rates_up, day_count_fx,
    us_calendar, ql.Linear(), ql.Continuous
)
usd_zc_up.enableExtrapolation()

df_usd_up   = usd_zc_up.discount(dt)
F_t_ir_up   = spot_eurusd * df_eur / df_usd_up
npv_ir_up   = notional_usd * (F_0 - F_t_ir_up) * df_eur
delta_ir    = npv_ir_up - npv_base

# analytical delta IR
delta_ir_analytical = -notional_usd * F_0 * T * df_eur * rate_bump

print(f"\nDelta IR (numerical, 1bp USD rate rise) : EUR {delta_ir:,.2f}")
print(f"Delta IR (analytical)                   : EUR {delta_ir_analytical:,.2f}")

print(f"\nFRTB SA risk classification:")
print(f"  Delta FX --> FX risk bucket  : EUR {delta_fx:,.2f} per 1% spot move")
print(f"  Delta IR --> GIRR bucket     : EUR {delta_ir:,.2f} per 1bp rate move")

Forward contract: deliver USD 1,000,000, receive EUR
Maturity        : 1Y
Contractual rate: 1.1792 EUR/USD

Current market forward : 1.1792
NPV (EUR)              : 0.00

Delta FX (numerical, 1% spot move) : EUR -11,555.10
Delta FX (analytical)              : EUR -9,980.22

Delta IR (numerical, 1bp USD rate rise) : EUR -117.16
Delta IR (analytical)                   : EUR -115.55

FRTB SA risk classification:
  Delta FX --> FX risk bucket  : EUR -11,555.10 per 1% spot move
  Delta IR --> GIRR bucket     : EUR -117.16 per 1bp rate move


## 6 Hedging a USD Portfolio -- Effectiveness Analysis

A Luxembourg fund holds USD 10M in US equities. The fund is EUR-based
and wants to eliminate FX risk by locking in the EUR value of the
portfolio using an FX forward.

The hedge ratio is straightforward -- sell USD 10M forward, receive
EUR at the CIP forward rate. A perfect hedge has zero delta FX after
the forward is added.

**Hedge effectiveness** is measured as the reduction in FX delta:

$$\text{Effectiveness} = 1 - \frac{|\Delta_{FX}^{hedged}|}{|\Delta_{FX}^{unhedged}|}$$

A perfect hedge gives effectiveness = 100%. In practice effectiveness
degrades over time as the forward notional diverges from the portfolio
value due to market moves -- requiring periodic rebalancing.

Under AIFMD II (Article 16) and UCITS risk management requirements,
funds must demonstrate and document hedging effectiveness for currency
hedged share classes. The forward must be sized to the portfolio NAV
and rebalanced at least monthly.

In [9]:
# -----------------------------------------------------------------------
# section 6 -- hedging a USD portfolio
# -----------------------------------------------------------------------

portfolio_usd = 10_000_000   # USD 10M equity portfolio
T_hedge       = 1.0          # hedge horizon 1 year
dt_hedge      = valuation_date + ql.Period(365, ql.Days)

# current forward rate for hedge
df_eur_h  = eur_zc.discount(dt_hedge)
df_usd_h  = usd_zc.discount(dt_hedge)
F_hedge   = spot_eurusd * df_eur_h / df_usd_h

# unhedged portfolio -- pure USD exposure
# delta FX of unhedged position = portfolio value in EUR at spot
portfolio_eur     = portfolio_usd / spot_eurusd
delta_fx_unhedged = portfolio_eur  # 1% spot move = 1% loss in EUR value

# forward hedge -- sell USD 10M forward
# delta FX of forward = -notional * F0/S0 * df_eur
delta_fx_forward  = -portfolio_usd * (F_hedge / spot_eurusd) * df_eur_h

# hedged portfolio delta FX
delta_fx_hedged   = delta_fx_unhedged + delta_fx_forward

# hedge effectiveness
effectiveness = (1 - abs(delta_fx_hedged) / abs(delta_fx_unhedged)) * 100

print(f"USD portfolio hedging analysis")
print(f"Portfolio value    : USD {portfolio_usd:,.0f} = EUR {portfolio_eur:,.0f}")
print(f"Hedge forward rate : {F_hedge:.4f} EUR/USD")
print(f"\n{'':35} {'Delta FX (EUR)':>15}")
print("-" * 52)
print(f"{'Unhedged portfolio (1% spot move)':35} {delta_fx_unhedged * 0.01:>15,.2f}")
print(f"{'Forward hedge (1% spot move)':35} {delta_fx_forward * 0.01:>15,.2f}")
print(f"{'Hedged portfolio (1% spot move)':35} {delta_fx_hedged * 0.01:>15,.2f}")
print(f"\nHedge effectiveness : {effectiveness:.2f}%")
print(f"\nAIFMD II requirement: rebalance monthly as NAV changes")
print(f"Current hedge ratio : {abs(delta_fx_forward)/abs(delta_fx_unhedged)*100:.2f}%")

USD portfolio hedging analysis
Portfolio value    : USD 10,000,000 = EUR 8,637,070
Hedge forward rate : 1.1792 EUR/USD

                                     Delta FX (EUR)
----------------------------------------------------
Unhedged portfolio (1% spot move)         86,370.70
Forward hedge (1% spot move)             -99,802.21
Hedged portfolio (1% spot move)          -13,431.50

Hedge effectiveness : 84.45%

AIFMD II requirement: rebalance monthly as NAV changes
Current hedge ratio : 115.55%


## 7 CIP Deviation -- The Cross-Currency Basis

In practice CIP does not hold exactly. The deviation is the
**cross-currency basis** $b$ -- an additional spread paid to obtain
foreign currency funding via the FX swap market:

$$F_t^{market} = S_t \cdot \frac{P_{EUR}(0,T)}{P_{USD}(0,T)} \cdot e^{b \cdot T}$$

A **negative basis** ($b < 0$) means USD funding via EUR FX swaps
costs more than CIP implies -- USD is scarce relative to EUR in the
swap market. The EUR/USD basis has been persistently negative since
2008, ranging from -10bps to -80bps depending on market stress.

**Why the basis exists:**

- **Regulatory demand for USD** -- non-US banks need USD to meet
  LCR requirements and fund USD assets
- **Hedging demand** -- EUR investors buying USD assets create
  structural demand for USD via FX swaps
- **Balance sheet constraints** -- post-Basel III, dealer banks
  have less capacity to arbitrage the basis away

The basis has direct P&L impact for any EUR-based institution
hedging USD exposure -- the actual cost of the hedge exceeds the
theoretical CIP rate by the basis spread.

Under EMIR reporting, cross-currency basis swaps are reportable
OTC derivatives and subject to margin requirements.

In [10]:
# -----------------------------------------------------------------------
# section 7 -- CIP deviation -- cross-currency basis
# -----------------------------------------------------------------------

# EUR/USD cross-currency basis -- persistently negative since 2008
# current approximate level -- in production from Bloomberg XCCY basis swap quotes
basis_bps = -20  # -20bps -- typical recent EUR/USD basis
basis     = basis_bps / 10000

print(f"Theoretical CIP forward rates vs basis-adjusted rates")
print(f"Cross-currency basis : {basis_bps}bps")
print(f"\n{'Tenor':<8} {'CIP forward':>12} {'Basis adj fwd':>15} {'Difference':>12} {'Cost (EUR)':>12}")
print("-" * 63)

notional_basis = 10_000_000  # EUR 10M hedging example

for label, T in forward_tenors.items():
    dt_b    = valuation_date + ql.Period(max(1, int(T * 365)), ql.Days)
    df_eur_b = eur_zc.discount(dt_b)
    df_usd_b = usd_zc.discount(dt_b)

    # CIP forward -- no basis
    fwd_cip  = spot_eurusd * df_eur_b / df_usd_b

    # basis-adjusted forward -- basis makes USD more expensive to obtain
    # negative basis means you receive fewer USD per EUR
    fwd_adj  = fwd_cip * np.exp(basis * T)

    # cost of basis for a hedger selling USD forward
    # hedger receives fwd_adj instead of fwd_cip per EUR notional
    cost_eur = notional_basis * (fwd_cip - fwd_adj) / spot_eurusd

    print(f"{label:<8} {fwd_cip:>12.4f} {fwd_adj:>15.4f} "
          f"{(fwd_adj-fwd_cip)*10000:>10.1f}bps {cost_eur:>12.2f}")

print(f"\nBasis cost interpretation:")
print(f"A EUR-based fund hedging USD {notional_basis:,.0f} for 1Y pays an additional")
fwd_1y     = spot_eurusd * eur_zc.discount(
    valuation_date + ql.Period(365, ql.Days)) / usd_zc.discount(
    valuation_date + ql.Period(365, ql.Days))
fwd_1y_adj = fwd_1y * np.exp(basis * 1.0)
cost_1y    = notional_basis * (fwd_1y - fwd_1y_adj) / spot_eurusd
print(f"EUR {cost_1y:,.0f} per year due to the {basis_bps}bps EUR/USD basis.")
print(f"This cost is not visible in the spot rate -- it only appears")
print(f"when hedging via the FX swap or forward market.")
print(f"\nHistorical EUR/USD basis range:")
print(f"  Normal markets  : -10 to -20bps")
print(f"  COVID stress    : -80bps (March 2020)")
print(f"  GFC peak        : -120bps (2008)")
print(f"  Current         : approx {basis_bps}bps")

Theoretical CIP forward rates vs basis-adjusted rates
Cross-currency basis : -20bps

Tenor     CIP forward   Basis adj fwd   Difference   Cost (EUR)
---------------------------------------------------------------
1M             1.1595          1.1593       -1.9bps      1668.93
3M             1.1630          1.1624       -5.8bps      5021.06
6M             1.1680          1.1668      -11.7bps     10082.70
1Y             1.1792          1.1769      -23.6bps     20349.52
2Y             1.2001          1.1953      -47.9bps     41377.96
5Y             1.2633          1.2507     -125.7bps    108569.20

Basis cost interpretation:
A EUR-based fund hedging USD 10,000,000 for 1Y pays an additional
EUR 20,350 per year due to the -20bps EUR/USD basis.
This cost is not visible in the spot rate -- it only appears
when hedging via the FX swap or forward market.

Historical EUR/USD basis range:
  Normal markets  : -10 to -20bps
  COVID stress    : -80bps (March 2020)
  GFC peak        : -120bps (2008)